# EMIT central-wavelength drift over time

Diagnostic: does EMIT's per-band **central wavelength** change across acquisitions?

- **Independent of our pipeline** — wavelengths are read straight from each granule's
  `sensor_band_parameters/wavelengths` group via `earthaccess` streaming, *not* from our
  per-channel CSVs.
- **Scope** — a single PICS/patch; only the granules that overlap this patch (taken from the
  patch–granule dictionary, `patch_EMIT_dict_{PICS}.csv`).
- **Plot** — x = acquisition date; y = **mean over all bands of (wavelength − first-scene
  wavelength)** in nm, with ±1 std whiskers (shows whether any drift is uniform across the
  spectrum). L1B RAD and L2A RFL overlaid.

Requires Earthdata credentials (same as the pipeline). Opening each granule reads only the
285-value wavelength array — no full-data download — but it is one CMR search + open per
granule per product, so expect a short wait.

In [3]:
# ============================================================
# USER-CONFIGURABLE INPUTS
# ============================================================
PICS_ID  = "Libya_4"     # underscored site id
PATCH_ID = "10_7_1"      # patch_id within the patch–granule dictionary

SAVE_FIG = True          # also write the figure to figures/ (gitignored)

In [4]:
import sys
from pathlib import Path

import numpy as np
import pandas as pd
import xarray as xr
import matplotlib.pyplot as plt
import earthaccess

# Locate the repo root (walk up until the emit_pics package is found), so the
# notebook runs whether opened from notebooks/ or the repo root.
_REPO_ROOT = Path.cwd()
while not (_REPO_ROOT / "emit_pics").is_dir() and _REPO_ROOT != _REPO_ROOT.parent:
    _REPO_ROOT = _REPO_ROOT.parent
sys.path.insert(0, str(_REPO_ROOT))

from emit_pics.id_matched_pairs import fetch_granule, sniff_delimiter
from emit_pics.extract_patch_stats import (
    load_patch_dict, split_products, pair_acquisitions, find_asset_url,
    RAD_TOKEN, RFL_TOKEN,
)

RAD_SHORT = "EMITL1BRAD"   # L1B radiance collection
RFL_SHORT = "EMITL2ARFL"   # L2A reflectance collection

DATA_ROOT = _REPO_ROOT / "data"
DICT_PATH = DATA_ROOT / "ROIs" / "patches" / f"patch_EMIT_dict_{PICS_ID}.csv"
print("repo root:", _REPO_ROOT)
print("dict:     ", DICT_PATH, "(exists)" if DICT_PATH.exists() else "(MISSING)")

repo root: C:\Users\huckr\Desktop\JPL\Code\emit-pics
dict:      C:\Users\huckr\Desktop\JPL\Code\emit-pics\data\ROIs\patches\patch_EMIT_dict_Libya_4.csv (MISSING)


In [5]:
# Authenticate to NASA Earthdata (persisted; same mechanism as the pipeline).
earthaccess.login(persist=True)
print("Authenticated to NASA Earthdata.")

Authenticated to NASA Earthdata.


In [6]:
# Overlapping granules for this patch, straight from the patch–granule dict,
# paired by acquisition (one RAD + one RFL per acquisition when both exist).
dict_df = load_patch_dict(DICT_PATH, sep=sniff_delimiter(DICT_PATH))
rad_urs, rfl_urs = split_products(dict_df, PATCH_ID)
acqs = pair_acquisitions(rad_urs, rfl_urs)
acqs = sorted(acqs, key=lambda a: a["datetime"])
print(f"Patch {PATCH_ID}: {len(rad_urs)} RAD, {len(rfl_urs)} RFL -> "
      f"{len(acqs)} acquisition(s)")

FileNotFoundError: [Errno 2] No such file or directory: 'C:\\Users\\huckr\\Desktop\\JPL\\Code\\emit-pics\\data\\ROIs\\patches\\patch_EMIT_dict_Libya_4.csv'

In [ ]:
def read_wavelengths(short_name: str, granule_ur: str, token: str) -> np.ndarray:
    """Central wavelengths (nm) read straight from a granule's
    sensor_band_parameters group — independent of any local processing."""
    gran = fetch_granule(short_name, granule_ur)
    url = find_asset_url(gran, token)
    fh = earthaccess.open([url])[0]
    try:
        ds = xr.open_dataset(fh, engine="h5netcdf", group="sensor_band_parameters")
        try:
            return np.asarray(ds["wavelengths"].values, dtype=float)
        finally:
            ds.close()
    finally:
        try:
            fh.close()
        except Exception:
            pass

In [ ]:
# Gather each scene's wavelength array for both products (chronological).
records = []
for k, acq in enumerate(acqs, 1):
    ts = acq["timestamp"]
    rec = {"datetime": acq["datetime"], "timestamp": ts}
    for label, short, token, ur in (
        ("RAD", RAD_SHORT, RAD_TOKEN, acq["rad_ur"]),
        ("RFL", RFL_SHORT, RFL_TOKEN, acq["rfl_ur"]),
    ):
        if ur is None:
            rec[label] = None
            continue
        try:
            rec[label] = read_wavelengths(short, ur, token)
        except Exception as ex:
            print(f"  [{k}/{len(acqs)}] {ts} {label} FAILED: {type(ex).__name__}: {ex}")
            rec[label] = None
    records.append(rec)
    n_rad = None if rec["RAD"] is None else len(rec["RAD"])
    n_rfl = None if rec["RFL"] is None else len(rec["RFL"])
    print(f"  [{k}/{len(acqs)}] {ts}  RAD bands={n_rad}  RFL bands={n_rfl}")

In [ ]:
def drift_series(records, label):
    """Per-scene mean & std of (wavelength − first-scene wavelength), across bands.

    Reference = the earliest scene that carries this product. Bands are compared
    by position (band i vs band i); if band counts ever differ the overlap is used.
    """
    ref = None
    ts, mean, std = [], [], []
    for r in records:
        wl = r[label]
        if wl is None:
            continue
        if ref is None:
            ref = wl
        n = min(len(wl), len(ref))
        d = wl[:n] - ref[:n]
        ts.append(r["datetime"])
        mean.append(float(np.mean(d)))
        std.append(float(np.std(d)))
    return ts, np.array(mean), np.array(std)

# Tidy summary table for eyeballing.
rows = []
for label in ("RAD", "RFL"):
    t, m, s = drift_series(records, label)
    for ti, mi, si in zip(t, m, s):
        rows.append({"product": label, "datetime": ti,
                     "mean_dwl_nm": mi, "std_dwl_nm": si})
summary = pd.DataFrame(rows).sort_values(["product", "datetime"]).reset_index(drop=True)
summary

In [ ]:
fig, ax = plt.subplots(figsize=(11, 5))
styles = {"RAD": ("tab:blue", "RAD (L1B)"), "RFL": ("tab:orange", "RFL (L2A)")}
for label, (color, legend) in styles.items():
    t, m, s = drift_series(records, label)
    if len(t):
        ax.errorbar(t, m, yerr=s, fmt="o-", capsize=3, color=color,
                    label=f"{legend}  (±1 std across bands)")
ax.axhline(0.0, color="gray", lw=0.8, ls="--")
ax.set_xlabel("Acquisition date")
ax.set_ylabel("Mean Δ central wavelength vs first scene (nm)")
ax.set_title(f"{PICS_ID} {PATCH_ID}: EMIT central-wavelength drift over time")
ax.legend()
fig.autofmt_xdate()
plt.tight_layout()

if SAVE_FIG:
    fig_dir = _REPO_ROOT / "figures"
    fig_dir.mkdir(parents=True, exist_ok=True)
    out = fig_dir / f"{PICS_ID}_{PATCH_ID}_wavelength_drift.png"
    fig.savefig(out, dpi=200, bbox_inches="tight")
    print(f"Saved -> {out}")
plt.show()

### Reading the plot

- **Flat at 0** — no wavelength drift; every scene shares the first scene's grid.
- **A step / offset** — EMIT recalibrated wavelengths between processing eras. A near-zero
  std whisker means the shift is **uniform across the whole spectrum** (a rigid offset, as
  seen for the 2025 Libya 4 granule at ~−0.088 nm); a large whisker would mean the shift is
  band-dependent.
- **RAD vs RFL overlapping** confirms the L1B and L2A grids agree for each acquisition.

This is why `delta_calculations.py` aligns paired scenes **by band order** rather than exact
wavelength: the channels correspond one-to-one even when the labels drift by a fraction of
the ~7.4 nm band spacing.